In [2]:
import polars as pl

In [6]:
df = pl.read_parquet("../data/results.parquet")

def process_row(row):
    col_names = row["col_name"]
    values = row["col_value"]
    attributes = [f"{col}={val}" for col, val in zip(col_names, values)]
    results = f"Samples with the following attributes: {', '.join(attributes)} had a mean concentration of totiviridae of {row['mean_positive']} and samples with different attributes had a mean concentration of totiviridae of {row['mean_negative']}."
    return results

# add a new column to the dataframe with the processed results
df = df.with_columns(
    pl.struct(["col_name", "col_value", "mean_positive", "mean_negative"])
    .map_elements(process_row, return_dtype=pl.String)
    .alias("processed_result")
)

In [ ]:
def build_hypothesis(row):
    return f"Analyzing SRA metadata and totiviridae abundance, we found the following statistical finding: {row['processed_result']} Please come up with a hypothesis that could explain this finding."

df = df.with_columns(
    pl.struct(["processed_result"])
    .map_elements(build_hypothesis, return_dtype=pl.String)
    .alias("prompt")
)

In [8]:
df.write_parquet("../data/results_with_hypotheses.parquet")